# Urban Heat Island — Nairobi: Data Exploration

Use this notebook to:
- Understand how Landsat thermal data is structured
- Visualise LST before building the Streamlit app
- Experiment with the analysis functions

Run cells top to bottom. Each cell has comments explaining what it does.

In [ ]:
# Standard imports
import sys, os
sys.path.insert(0, '../src')   # So Python can find our src/ modules

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from config import AOI_BBOX, AOI_NAME
from analysis import (
    make_simple_urban_mask,
    compute_uhi_intensity,
    detect_hotspots,
    detect_cool_islands,
    compute_temperature_change,
)

print('Imports OK')

## Step 1 — Load LST data

If you have already run `src/data_fetch.py`, load the real GeoTIFF.
Otherwise the cell below generates synthetic data for you to experiment with.

In [ ]:
import rasterio

def load_or_simulate(year):
    path = f'../data/raw/lst_{year}.tif'
    if os.path.exists(path):
        print(f'Loading real data from {path}')
        with rasterio.open(path) as src:
            arr = src.read(1).astype(np.float32)
            arr[arr == src.nodata] = np.nan
        return arr
    else:
        print(f'No file at {path} — using synthetic data')
        rng = np.random.default_rng({'2015': 1, '2023': 2}.get(str(year), 0))
        base = rng.normal(30, 3, (200, 220)).astype(np.float32)
        if str(year) == '2023': base += 1.5
        base[60:100,  90:130] += 7
        base[90:130, 130:170] += 5
        base[20:55,   10:60]  -= 6
        base[130:180, 30:90]  -= 5
        return base

lst_2015 = load_or_simulate(2015)
lst_2023 = load_or_simulate(2023)

print(f'Shape: {lst_2023.shape}')
print(f'Min: {np.nanmin(lst_2023):.1f}°C  |  Max: {np.nanmax(lst_2023):.1f}°C')

## Step 2 — Visualise LST as a heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, lst, year in zip(axes, [lst_2015, lst_2023], [2015, 2023]):
    vmin, vmax = np.nanpercentile(lst, [2, 98])   # clip outliers for display
    im = ax.imshow(lst, cmap='RdYlBu_r', vmin=vmin, vmax=vmax)
    ax.set_title(f'LST — {year} dry season', fontsize=13)
    ax.axis('off')
    plt.colorbar(im, ax=ax, label='°C', shrink=0.8)

plt.suptitle(f'Land Surface Temperature — {AOI_NAME}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Step 3 — Compute UHI intensity

In [ ]:
for lst, year in [(lst_2015, 2015), (lst_2023, 2023)]:
    mask = make_simple_urban_mask(lst)
    stats = compute_uhi_intensity(lst, mask)
    print(f'\n--- {year} ---')
    print(f'Urban mean:   {stats["urban_mean_celsius"]:.2f}°C')
    print(f'Rural mean:   {stats["rural_mean_celsius"]:.2f}°C')
    print(f'UHI intensity: {stats["uhi_intensity"]:.2f}°C')

## Step 4 — Temperature change map (2015 → 2023)

In [ ]:
change = compute_temperature_change(lst_2015, lst_2023)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(change, cmap='RdBu_r', vmin=-5, vmax=5)
ax.set_title('Temperature change 2015 → 2023 (°C)', fontsize=13)
ax.axis('off')
plt.colorbar(im, ax=ax, label='Δ°C')
plt.tight_layout()
plt.show()

print(f'Mean change: {np.nanmean(change):.2f}°C')
print(f'Pixels that warmed >2°C: {(change > 2).sum():,}')

## Step 5 — Hot spots and cool islands

In [ ]:
hotspots = detect_hotspots(lst_2023)
cool     = detect_cool_islands(lst_2023)

# Overlay on the LST map
rgb = plt.cm.RdYlBu_r((lst_2023 - np.nanmin(lst_2023)) / (np.nanmax(lst_2023) - np.nanmin(lst_2023)))

fig, ax = plt.subplots(figsize=(8, 7))
ax.imshow(rgb)

# Overlay hot spot contours in red, cool island contours in blue
ax.contour(hotspots, levels=[0.5], colors=['#D85A30'], linewidths=1.5)
ax.contour(cool,     levels=[0.5], colors=['#185FA5'], linewidths=1.5)

ax.set_title('Hot spots (orange) and cool islands (blue) — 2023', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()